In [4]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 147 (delta 56), reused 123 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 42.69 MiB | 22.33 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [3]:
from huggingface_hub import login
login()

In [6]:
%cd Dual_watermarking_Scheme/

/content/Dual_watermarking_Scheme


In [7]:
!ls

data  model  notebooks	README.md  requirements.txt  setup.md  src


In [8]:
!pip uninstall -y transformers peft accelerate datasets bitsandbytes

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0


In [9]:
!pip install -U \
transformers==5.15.0 \
peft==0.20.0 \
accelerate==1.14.0 \
datasets==5.0.1 \
bitsandbytes==0.50.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 114.9 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [11]:
import torch
import bitsandbytes,transformers,peft,accelerate

print(torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

2.11.0+cu128
transformers: 5.15.0
peft: 0.20.0
accelerate: 1.14.0
bitsandbytes: 0.50.1


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [14]:
from transformers import BitsAndBytesConfig
import bitsandbytes as bnb

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("BitsAndBytesConfig created successfully:")
print(bnb_config)

print("\nbitsandbytes CUDA setup check:")
print(bnb.__version__)

BitsAndBytesConfig created successfully:
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


bitsandbytes CUDA setup check:
0.50.1


In [15]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import PeftModel

BASE_MODEL = "facebook/opt-2.7b"
adapter_path = "./model/v1"


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)


model = PeftModel.from_pretrained(
    model,
    adapter_path
)

model.eval()


prompt = """
Topic: Probability
Difficulty: Hard
Question:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )


print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


Topic: Probability
Difficulty: Hard
Question:
What is the probability that two events occurring simultaneously have the same probability, and how does this relate to the concept of a conditional probability?


In [16]:
from src.utils.loadConfig import load_config
config = load_config("secondLayer")

config

{'MODEL_NAME': 'facebook/opt-2.7b',
 'GREEN_FRACTION': 0.5,
 'PREV_TOKEN_SIZE': 5,
 'DETECTION_THRESHOLD': 0.6,
 'P_VALUE_THRESHOLD': 0.05}

In [17]:
from src.utils.key_manager import generate_key  

key = generate_key()
key

'2331514dcbf64a9110d7541854e67647025dc1065a436c874d53418f633359f0'

In [18]:
from src.watermark.second_layer import PrivateWatermarkProcessor

processor = PrivateWatermarkProcessor(
    key=key,
    vocab_size=len(tokenizer),
    green_fraction=config["GREEN_FRACTION"],
    delta_private=0.7,
    prev_token_size=config["PREV_TOKEN_SIZE"]
)

print("watermark processor loaded .")

watermark processor loaded .


In [19]:
from src.watermark.second_layer import generation_pipeline

prompt = """
Topic: Probability
Difficulty: Hard
Question:
"""


results = generation_pipeline(
    prompts=[prompt],
    model=model,
    tokenizer=tokenizer,
    processors=[processor],
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)


In [20]:
results

,id,prompt,plain_output,watermarked_output
0,0,\nTopic: Probability\nDifficulty: Hard\nQuesti...,\nTopic: Probability\nDifficulty: Hard\nQuesti...,\nTopic: Probability\nDifficulty: Hard\nQuesti...


In [21]:
str = results['plain_output'][0]

print("plain output generation ------ \n")
print(str)

print("watermarked output ------ \n")
str = results['watermarked_output'][0]
print(str)

plain output generation ------ 


Topic: Probability
Difficulty: Hard
Question:
How does the fact that P(a|b) is true for all a and b ensure that the agent's behavior is consistent with the belief structure, and what does this mean for the agent's belief structure when P(a|b) is false?
watermarked output ------ 


Topic: Probability
Difficulty: Hard
Question:
What is the role of the conditional probability of a variable in the Bayesian network, and how does this probability differ from the exact probability of the variable because of the non-monotonic relationship between the variable and the conditional probability of the other variables in the network?


In [23]:
print(type(key))
print(key)

<class 'str'>
2331514dcbf64a9110d7541854e67647025dc1065a436c874d53418f633359f0


In [24]:
from src.watermark.second_layer import PrivateWatermarkProcessor, generation_pipeline
from src.detection.kgw_detection import detect_private_watermark

test_key = key  


processor = PrivateWatermarkProcessor(
    key=test_key,
    vocab_size=len(tokenizer),
    green_fraction=config["GREEN_FRACTION"],
    delta_private=0.7,
    prev_token_size=config["PREV_TOKEN_SIZE"]
)


prompt = """
Topic: Probability
Difficulty: Hard
Question:
"""


results = generation_pipeline(
    prompts=[prompt],
    model=model,
    tokenizer=tokenizer,
    processors=[processor],
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)


watermarked_text = results.iloc[0]["watermarked_output"]


print("========== GENERATED TEXT ==========")
print(watermarked_text)


detection_result = detect_private_watermark(
    text=watermarked_text,
    tokenizer=tokenizer,
    key=test_key,
    vocab_size=len(tokenizer),
    green_fraction=config["GREEN_FRACTION"],
    prev_token_size=config["PREV_TOKEN_SIZE"],
    threshold=config["DETECTION_THRESHOLD"],
    p_value_threshold=config["P_VALUE_THRESHOLD"],
)


print("\n========== DETECTION RESULT ==========")

for k, v in detection_result.items():
    print(f"{k}: {v}")

========== GENERATED TEXT ==========

Topic: Probability
Difficulty: Hard
Question:
What are the four main components of the Bayesian network for estimating conditional probabilities, and how do these components connect to the variables' causal relationships in the context of the vacuum cleaner example?

========== DETECTION RESULT ==========
ownership_score: 0.5869565217391305
match_count: 27
num_positions: 46
p_value: 0.1509978065948303
confirmed: False


In [25]:
key

'2331514dcbf64a9110d7541854e67647025dc1065a436c874d53418f633359f0'

In [ ]:
import gradio as gr

from src.watermark.second_layer import PrivateWatermarkProcessor, generation_pipeline
from src.detection.kgw_detection import detect_private_watermark

def generate_question(topic, difficulty):

    prompt = f"""
    Topic: {topic}
    Difficulty: {difficulty}
    Question:
"""

    test_key = key

    processor = PrivateWatermarkProcessor(
        key=test_key,
        vocab_size=len(tokenizer),
        green_fraction=config["GREEN_FRACTION"],
        delta_private=0.9,
        prev_token_size=config["PREV_TOKEN_SIZE"]
    )

    results = generation_pipeline(
        prompts=[prompt],
        model=model,
        tokenizer=tokenizer,
        processors=[processor],
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

    watermarked_text = results.iloc[0]["watermarked_output"]
    return watermarked_text


def detect_question(text, secret_key):

    result = detect_private_watermark(
        text=text,
        tokenizer=tokenizer,
        key=secret_key,
        vocab_size=len(tokenizer),
        green_fraction=config["GREEN_FRACTION"],
        prev_token_size=config["PREV_TOKEN_SIZE"],
        threshold=config["DETECTION_THRESHOLD"],
        p_value_threshold=config["P_VALUE_THRESHOLD"],
    )

    return (
        f"Ownership score: {result['ownership_score']:.4f}\n"
        f"Matched tokens: {result['match_count']}\n"
        f"Total positions: {result['num_positions']}\n"
        f"P-value: {result['p_value']:.6f}\n"
        f"Confirmed: {result['confirmed']}"
    )


with gr.Blocks(title="Private Watermark LLM") as app:

    gr.Markdown(
        """
        # Private Watermark LLM
        
        Generate watermark-protected questions and verify ownership.
        """
    )

    current_key = gr.Textbox(
        value=key,
        label="secret key"
    )


    with gr.Tab("Generate"):

        topic = gr.Dropdown(
            choices=[
                "Probability",
                "Artificial Intelligence",
                "Machine Learning",
                "Mathematics",
                "Physics",
                "Computer Science"
            ],
            value="Probability",
            label="Topic"
        )

        difficulty = gr.Dropdown(
            choices=[
                "Easy",
                "Medium",
                "Hard"
            ],
            value="Hard",
            label="Difficulty"
        )

        generate_button = gr.Button("Generate Question")

        generated_output = gr.Textbox(
            label="Watermarked Question",
            lines=12
        )


        generate_button.click(
            fn=generate_question,
            inputs=[
                topic,
                difficulty
            ],
            outputs=generated_output
        )


    with gr.Tab("Detect"):

        question_input = gr.Textbox(
            label="Question Text",
            placeholder="Paste generated question here...",
            lines=12
        )


        secret_key_input = gr.Textbox(
            label="Secret Key",
        )


        detect_button = gr.Button("Detect Watermark")


        detection_output = gr.Textbox(
            label="Detection Result",
            lines=8
        )


        detect_button.click(
            fn=detect_question,
            inputs=[
                question_input,
                secret_key_input
            ],
            outputs=detection_output
        )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90a6a4c629b2e0b09e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
